                                    TestCase scenario using NLP and RAG (healthcare payer domain)

                                                            Explanation
Rule Preparation-
I start with predefined raw business rules stored in a file.
These rules are chunked into 99 smaller, meaningful rule segments so that retrieval works at rule level instead of document level.

Vectorization-
Each rule chunk is converted into a TF-IDF vector.
This allows textual rules to be represented numerically for similarity comparison.

Query Processing-
When a user or tester asks a question, the query is also converted into a TF-IDF vector using the same vectorizer.

Rule Retrieval-
Using cosine similarity, the query vector is compared against all rule chunk vectors.
The system retrieves the top-matching rule chunk based on the highest similarity score.

Rule Parsing-
The retrieved rule text is parsed into a structured format, extracting:
Conditions
Coverage logic
Constraints
This converts unstructured rule text into machine-understandable JSON.

Test Case Generation-
Based on the structured rule JSON, the system automatically generates:
Preconditions
Test steps
Expected results
This removes manual test-case writing.

Tool Integration-
The generated test cases are exported and integrated with qTest and Katalon, enabling:
Test management
Automation execution

┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│  Raw Benefits   │    │   99 TF-IDF      │    │  QTest/Katlon   │
│  File (Excel)   │───▶│  Chunks (Vector  │───▶│  Test Execution │
└─────────────────┘    │  Store)          │    └─────────────────┘
                        └──────────────────┘
                              │
                       ┌──────────────┐
                       │  USER QUERY  │
                       │"Platinum PCP"│
                       └──────────────┘
                              │
                        ┌──────────────┐
                        │ COSINE MATCH │──┐
                        │   0.69 score │  │
                        └──────────────┘  │
                              │          │
                        ┌──────────────┐  │
                        │  PARSE RULE  │  │
                        │ → JSON       │  │
                        └──────────────┘  │
                              │          │
                        ┌──────────────┐  │
                        │ TEST CASE    │◄─┘
                        │ JSON → QTest │
                        └──────────────┘


In [3]:
from pathlib import Path
import os
import pandas as pd

# Step 1: ensure correct working directory
os.chdir(r"C:\Users\shine")

# Step 2: build path safely
DATA_DIR = Path("data")
BENEFITS_PATH = DATA_DIR / "synthetic_company_plan_benefits.csv"

# Step 3: validate
assert BENEFITS_PATH.exists(), "Data file not found"

# Step 4: load
df = pd.read_csv(BENEFITS_PATH)
df.head()

,plan_name,metal_level,deductible_individual,deductible_family,oop_max_individual,oop_max_family,hsa_compatible,primary_care_copay,specialist_copay,urgent_care_copay,er_copay,mental_health_copay,rx_tier1_copay,rx_tier2_copay,rx_tier3_copay,rx_tier4_copay
0,Platinum Classic,Platinum,0,0,2000,4000,No,15,35,55,100,15,10,30,60,60
1,Gold Classic,Gold,775,1550,10150,20300,No,25,40,60,150,25,10,35,70,70
2,Gold Simple,Gold,1500,3000,7200,14400,No,30,20,20,20,20,20,20,20,20
3,Silver Classic,Silver,2450,4900,10150,20300,No,30,65,70,500,30,15,40,75,75
4,Silver Simple PCP Saver,Silver,7300,14600,10100,20200,No,15,40,75,50,15,20,50,50,50


In [4]:
print(df.columns.tolist())

['plan_name', 'metal_level', 'deductible_individual', 'deductible_family', 'oop_max_individual', 'oop_max_family', 'hsa_compatible', 'primary_care_copay', 'specialist_copay', 'urgent_care_copay', 'er_copay', 'mental_health_copay', 'rx_tier1_copay', 'rx_tier2_copay', 'rx_tier3_copay', 'rx_tier4_copay']


In [7]:
service_map = {
    "Primary Care Visit": "primary_care_copay",
    "Specialist Visit": "specialist_copay",
    "Urgent Care": "urgent_care_copay",
    "Emergency Room": "er_copay",
    "Mental Health Visit": "mental_health_copay",
    "RX Tier 1": "rx_tier1_copay",
    "RX Tier 2": "rx_tier2_copay",
    "RX Tier 3": "rx_tier3_copay",
    "RX Tier 4": "rx_tier4_copay"
}

In [6]:
rag_chunks = []

for _, row in df.iterrows():
    for service_name, col_name in service_map.items():
        cost_share = row[col_name]

        chunk_text = (
            f"Plan: {row['plan_name']}\n"
            f"Metal Level: {row['metal_level']}\n"
            f"Service: {service_name}\n"
            f"Cost Share: {cost_share}\n"
            f"Individual Deductible: {row['deductible_individual']}\n"
            f"Family Deductible: {row['deductible_family']}\n"
            f"OOP Max (Individual): {row['oop_max_individual']}\n"
            f"OOP Max (Family): {row['oop_max_family']}\n"
            f"HSA Compatible: {row['hsa_compatible']}"
        )

        metadata = {
            "plan": row["plan_name"],
            "metal": row["metal_level"],
            "service": service_name,
            "year": "2026",
            "source": "Synthetic Benefits CSV"
        }

        rag_chunks.append({
            "text": chunk_text,
            "metadata": metadata
        })

len(rag_chunks)


99

This code creates normalized RAG chunks with metadata from structured CSV data. 

In [8]:
rag_chunks[0]


{'text': 'Plan: Platinum Classic\nMetal Level: Platinum\nService: Primary Care Visit\nCost Share: 15\nIndividual Deductible: 0\nFamily Deductible: 0\nOOP Max (Individual): 2000\nOOP Max (Family): 4000\nHSA Compatible: No',
 'metadata': {'plan': 'Platinum Classic',
  'metal': 'Platinum',
  'service': 'Primary Care Visit',
  'year': '2026',
  'source': 'Synthetic Benefits CSV'}}

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build corpus
rag_texts = [c["text"] for c in rag_chunks]

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(rag_texts)

X.shape

(99, 90)

99 healthcare rule chunks were vectorized into a 90-dimensional feature space.

In [10]:
def retrieve_benefit_rule(query, top_k=1):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, X)[0]

    top_idx = scores.argsort()[-top_k:][::-1]

    results = []
    for idx in top_idx:
        results.append({
            "score": scores[idx],
            "text": rag_chunks[idx]["text"],
            "metadata": rag_chunks[idx]["metadata"]
        })
    return results

In [12]:
query = "Platinum Classic primary care visit cost"
result = retrieve_benefit_rule(query)[0]

result


{'score': np.float64(0.6884786293766466),
 'text': 'Plan: Platinum Classic\nMetal Level: Platinum\nService: Primary Care Visit\nCost Share: 15\nIndividual Deductible: 0\nFamily Deductible: 0\nOOP Max (Individual): 2000\nOOP Max (Family): 4000\nHSA Compatible: No',
 'metadata': {'plan': 'Platinum Classic',
  'metal': 'Platinum',
  'service': 'Primary Care Visit',
  'year': '2026',
  'source': 'Synthetic Benefits CSV'}}

This step retrieves the most relevant insurance rule chunk from the RAG corpus using TF-IDF and cosine similarity based on the user query.

In [14]:
synthetic_member = {
    "member_id": "SYN-001",
    "age": 35,
    "state": "NY",
    "plan": "Platinum Classic"
}

synthetic_claim = {
    "claim_id": "CLM-1001",
    "service": "Primary Care Visit",
    "billed_amount": 150,
    "provider_network": "INN"
}


I use synthetic member and claim objects to simulate real-world insurance scenarios and validate retrieved benefit rules before generating automated test cases.

In [16]:
def parse_rag_rule(rag_text):
    rule = {}
    for line in rag_text.split("\n"):
        if line.startswith("Cost Share:"):
            rule["cost_share"] = int(
                line.replace("Cost Share:", "").strip()
            )
        if line.startswith("Individual Deductible:"):
            rule["deductible_individual"] = int(
                line.replace("Individual Deductible:", "").strip()
            )
        if line.startswith("HSA Compatible:"):
            rule["hsa"] = (
                line.replace("HSA Compatible:", "").strip().lower() == "yes"
            )
    return rule



I parse the retrieved RAG rule text into a normalized, typed structure so downstream decision logic and automated test cases can be generated reliably.

In [17]:
retrieved = retrieve_benefit_rule(
    "Platinum Classic primary care visit cost"
)[0]

parsed_rule = parse_rag_rule(retrieved["text"])
parsed_rule

{'cost_share': 15, 'deductible_individual': 0, 'hsa': False}

In [18]:
test_case = {
    "Test Case Name": "NY_2026_Platinum_PCP_CostShare",
    "Requirement": "Benefit Cost Sharing Validation",
    "Precondition": (
        f"Synthetic member ({synthetic_member['member_id']}) enrolled in "
        f"{synthetic_member['plan']} plan"
    ),
    "Step 1": (
        f"Submit an in-network {synthetic_claim['service']} claim "
        f"with billed amount ${synthetic_claim['billed_amount']}"
    ),
    "Expected Result": (
        f"Apply cost share of {parsed_rule['cost_share']} with "
        f"individual deductible {parsed_rule['deductible_individual']}"
    ),
    "Tags": "NY,2026,Platinum,Synthetic,RAG"
}

test_case


{'Test Case Name': 'NY_2026_Platinum_PCP_CostShare',
 'Requirement': 'Benefit Cost Sharing Validation',
 'Precondition': 'Synthetic member (SYN-001) enrolled in Platinum Classic plan',
 'Step 1': 'Submit an in-network Primary Care Visit claim with billed amount $150',
 'Expected Result': 'Apply cost share of 15 with individual deductible 0',
 'Tags': 'NY,2026,Platinum,Synthetic,RAG'}

After retrieving and parsing the benefit rule, I combine synthetic member and claim data to generate a structured, tool-ready test case that can be exported directly to qTest or Katalon

In [19]:
import pandas as pd
from pathlib import Path

df_qtest = pd.DataFrame([test_case])

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

csv_path = OUT_DIR / "qtest_testcases.csv"
df_qtest.to_csv(csv_path, index=False)

csv_path



WindowsPath('output/qtest_testcases.csv')

In [28]:
from pathlib import Path

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

csv_path = OUTPUT_DIR / "qtest_testcases.csv"
csv_path


WindowsPath('C:/Users/shine/output/qtest_testcases.csv')

In [29]:
KATALON_TEMPLATE = """
import com.kms.katalon.core.webui.keyword.WebUiBuiltInKeywords as WebUI

WebUI.comment("{test_id}")
WebUI.comment("{description}")
WebUI.comment("Expected: {expected}")
"""

In [30]:
from pathlib import Path

def generate_katalon_test(test_case, katalon_testcase_path):
    content = KATALON_TEMPLATE.format(
        test_id=test_case["test_id"],
        description=test_case["description"],
        expected=test_case["expected"].replace("$", "\\$")
    )

    file_path = Path(katalon_testcase_path) / f"{test_case['test_id']}.groovy"
    file_path.write_text(content, encoding="utf-8")

    return file_path


A Groovy test case template is a reusable script skeleton with placeholders, and test automation scaffolding refers to the supporting structure that allows these templates to be programmatically filled and generated as executable test cases.

In [31]:
test_case = {
    "test_id": "TC_Platinum_PCP",
    "description": "Validate Platinum Classic – Primary Care Visit cost sharing",
    "expected": "$15 copay, no deductible applies"
}

KATALON_OUT = Path("katalon/Test Cases")
KATALON_OUT.mkdir(parents=True, exist_ok=True)

generate_katalon_test(test_case, KATALON_OUT)




WindowsPath('katalon/Test Cases/TC_Platinum_PCP.groovy')

Currently the system generates one test case because it processes a single query with top-1 retrieval. Expanding the query set or increasing top-k allows automatic generation of multiple test cases